# Objective 1 — Identify High-Risk Patients Likely to Be Readmitted

In [24]:
#imports
import pandas as pd
import numpy as np
import joblib
from pathlib import Path
try:
    import sklearn
except ModuleNotFoundError:
    %pip install scikit-learn scipy

from sklearn.model_selection import train_test_split, StratifiedKFold, GridSearchCV, RandomizedSearchCV
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier, AdaBoostClassifier
from sklearn.utils.class_weight import compute_sample_weight
from scipy.stats import randint, uniform
from sklearn.metrics import ConfusionMatrixDisplay

try:
    from xgboost import XGBClassifier
except ModuleNotFoundError:
    %pip install xgboost
    from xgboost import XGBClassifier

try:
    from catboost import CatBoostClassifier
except ModuleNotFoundError:
    %pip install catboost
    from catboost import CatBoostClassifier

In [25]:
# Configuration for random state, data paths, and output directories
RANDOM_STATE = 42
MODEL_NAMES = ["logistic_regression", "random_forest", "gradient_boosting", "adaboost", "xgboost", "catboost"]
RISK_TABLE_PATH = Path("../temp/goal1_risk_scored_patients.csv")

DATA_PATH = Path("../temp/cleaned_dataSet.csv")

MODEL_DIR = Path("../temp/model")
MODEL_DIR.mkdir(parents=True, exist_ok=True)

OUTPUT_DIR = Path("../output")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

In [26]:
df = pd.read_csv(DATA_PATH)
print(df.shape)
print(df.dtypes)


(44314, 27)
patient_id                          int64
age                                 int64
gender                                str
bmi                               float64
smoking_status                        str
diabetes_flag                       int64
hypertension_flag                   int64
heart_disease_flag                  int64
chronic_conditions_count            int64
previous_admissions_12m             int64
length_of_stay_days                 int64
icu_admission_flag                  int64
emergency_admission_flag            int64
number_of_procedures                int64
blood_glucose                     float64
cholesterol_level                 float64
hemoglobin                        float64
creatinine                        float64
medications_count                   int64
high_risk_medication_flag           int64
medication_changes_during_stay      int64
followup_scheduled_flag             int64
discharge_destination                 str
patient_education_scor

## Features and target


In [27]:
TARGET = "readmission_flag"
DROP_COLS = ["patient_id", TARGET]

feature_cols = [c for c in df.columns if c not in DROP_COLS]
X = df[feature_cols].copy()
y = df[TARGET].copy()

categorical_cols = X.select_dtypes(include=["object", "str"]).columns.tolist()
numeric_cols = X.select_dtypes(exclude=["object", "str"]).columns.tolist()

binary_cols = [
    c for c in numeric_cols
    if X[c].dropna().nunique() <= 2 and set(X[c].dropna().unique()).issubset({0, 1})
]

 

print("n_features:", len(feature_cols))
print("categorical_cols:", categorical_cols)
print("numeric_cols:", numeric_cols)
print("target distribution (train pool):")
print(y.value_counts(normalize=True))


n_features: 25
categorical_cols: ['gender', 'smoking_status', 'discharge_destination', 'insurance_type']
numeric_cols: ['age', 'bmi', 'diabetes_flag', 'hypertension_flag', 'heart_disease_flag', 'chronic_conditions_count', 'previous_admissions_12m', 'length_of_stay_days', 'icu_admission_flag', 'emergency_admission_flag', 'number_of_procedures', 'blood_glucose', 'cholesterol_level', 'hemoglobin', 'creatinine', 'medications_count', 'high_risk_medication_flag', 'medication_changes_during_stay', 'followup_scheduled_flag', 'patient_education_score', 'treatment_cost']
target distribution (train pool):
readmission_flag
0    0.653992
1    0.346008
Name: proportion, dtype: float64


In [28]:
#Check if X is having missing value
for value in X.columns:
    if X[value].isna().any():
        print(value)

#Earlier creatine was missed in this step, so do not remove this check

# Train/Test Split

In [29]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    stratify=y,
    random_state=RANDOM_STATE,
)

print("Train shape:", X_train.shape, "Test shape:", X_test.shape)
print("Train class ratio:\n", y_train.value_counts(normalize=True))
print("Test class ratio:\n", y_test.value_counts(normalize=True))


Train shape: (35451, 25) Test shape: (8863, 25)
Train class ratio:
 readmission_flag
0    0.654001
1    0.345999
Name: proportion, dtype: float64
Test class ratio:
 readmission_flag
0    0.653955
1    0.346045
Name: proportion, dtype: float64


## Preprocessing Pipeline

In [30]:
numeric_transformer = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
])

binary_transformer = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
])

categorical_transformer = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore")),
])

preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, numeric_cols),
        ("bin", binary_transformer, binary_cols),
        ("cat", categorical_transformer, categorical_cols),
    ]
)


## Model Candidates

In [31]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
SCORING = "roc_auc"

search_results = {}


## 1 Logistic Regression

In [32]:
# Logistic Regression (baseline)
lr_pipeline = Pipeline([
    ("preprocessor", preprocessor),
    ("clf", LogisticRegression(class_weight="balanced", max_iter=1000, random_state=RANDOM_STATE)),
])

lr_param_grid = {
    "clf__C": [0.01, 0.1, 1.0, 10.0],
    "clf__solver": ["lbfgs"],
}

lr_search = GridSearchCV(lr_pipeline, lr_param_grid, scoring=SCORING, cv=cv, n_jobs=-1)
lr_search.fit(X_train, y_train)

search_results["logistic_regression"] = lr_search
print("Logistic Regression best CV ROC AUC:", lr_search.best_score_)
print("Logistic Regression best params:", lr_search.best_params_)


Logistic Regression best CV ROC AUC: 0.8818560619764708
Logistic Regression best params: {'clf__C': 0.01, 'clf__solver': 'lbfgs'}


## 2 Random Forest

In [33]:
# Random Forest
rf_pipeline = Pipeline([
    ("preprocessor", preprocessor),
    ("clf", RandomForestClassifier(class_weight="balanced", random_state=RANDOM_STATE, n_jobs=-1)),
])

rf_param_dist = {
    "clf__n_estimators": randint(100, 400),
    "clf__max_depth": [None, 5, 10, 20],
    "clf__min_samples_leaf": randint(1, 6),
    "clf__max_features": ["sqrt", "log2"],
}

rf_search = RandomizedSearchCV(
    rf_pipeline, rf_param_dist, n_iter=15, scoring=SCORING, cv=cv,
    random_state=RANDOM_STATE, n_jobs=-1,
)
rf_search.fit(X_train, y_train)

search_results["random_forest"] = rf_search
print("Random Forest best CV ROC AUC:", rf_search.best_score_)
print("Random Forest best params:", rf_search.best_params_)


Random Forest best CV ROC AUC: 0.9250722054877851
Random Forest best params: {'clf__max_depth': None, 'clf__max_features': 'sqrt', 'clf__min_samples_leaf': 5, 'clf__n_estimators': 266}


## 3 Gradient Boosting

In [ ]:
# Gradient Boosting
gb_pipeline = Pipeline([
    ("preprocessor", preprocessor),
    ("clf", GradientBoostingClassifier(random_state=RANDOM_STATE)),
])

gb_param_dist = {
    "clf__n_estimators": randint(100, 400),
    "clf__learning_rate": uniform(0.01, 0.29),
    "clf__max_depth": randint(2, 6),
    "clf__subsample": uniform(0.7, 0.3),
}


sample_weight = compute_sample_weight(class_weight="balanced", y=y_train)

gb_search = RandomizedSearchCV(
    gb_pipeline, gb_param_dist, n_iter=15, scoring=SCORING, cv=cv,
    random_state=RANDOM_STATE, n_jobs=-1,
)
gb_search.fit(X_train, y_train, clf__sample_weight=sample_weight)

search_results["gradient_boosting"] = gb_search
print("Gradient Boosting best CV ROC AUC:", gb_search.best_score_)
print("Gradient Boosting best params:", gb_search.best_params_)


## 4 AdaBoost


In [ ]:
# AdaBoost
adaboost_pipeline = Pipeline([
    ("preprocessor", preprocessor),
    ("clf", AdaBoostClassifier(random_state=RANDOM_STATE)),
])

adaboost_param_dist = {
    "clf__n_estimators": randint(50, 300),
    "clf__learning_rate": uniform(0.01, 1.49),
}


adaboost_search = RandomizedSearchCV(
    adaboost_pipeline, adaboost_param_dist, n_iter=10, scoring=SCORING, cv=cv,
    random_state=RANDOM_STATE, n_jobs=-1,
)
adaboost_search.fit(X_train, y_train, clf__sample_weight=sample_weight)

search_results["adaboost"] = adaboost_search
print("AdaBoost best CV ROC AUC:", adaboost_search.best_score_)
print("AdaBoost best params:", adaboost_search.best_params_)


## 5 XGBoost

In [ ]:
# XGBoost
xgb_pipeline = Pipeline([
    ("preprocessor", preprocessor),
    ("clf", XGBClassifier(random_state=RANDOM_STATE, eval_metric="logloss", n_jobs=-1)),
])

xgb_param_dist = {
    "clf__n_estimators": randint(100, 400),
    "clf__max_depth": randint(3, 8),
    "clf__learning_rate": uniform(0.01, 0.29),
    "clf__subsample": uniform(0.7, 0.3),
    "clf__colsample_bytree": uniform(0.7, 0.3),
}


xgb_search = RandomizedSearchCV(
    xgb_pipeline, xgb_param_dist, n_iter=10, scoring=SCORING, cv=cv,
    random_state=RANDOM_STATE, n_jobs=-1,
)
xgb_search.fit(X_train, y_train, clf__sample_weight=sample_weight)

search_results["xgboost"] = xgb_search
print("XGBoost best CV ROC AUC:", xgb_search.best_score_)
print("XGBoost best params:", xgb_search.best_params_)


## 6 CatBoost

In [ ]:
# CatBoost
catboost_pipeline = Pipeline([
    ("preprocessor", preprocessor),
    ("clf", CatBoostClassifier(random_state=RANDOM_STATE, verbose=0, allow_writing_files=False)),
])

catboost_param_dist = {
    "clf__iterations": randint(100, 400),
    "clf__depth": randint(3, 8),
    "clf__learning_rate": uniform(0.01, 0.29),
    "clf__l2_leaf_reg": uniform(1, 9),
}


catboost_search = RandomizedSearchCV(
    catboost_pipeline, catboost_param_dist, n_iter=10, scoring=SCORING, cv=cv,
    random_state=RANDOM_STATE, n_jobs=-1,
)
catboost_search.fit(X_train, y_train, clf__sample_weight=sample_weight)

search_results["catboost"] = catboost_search
print("CatBoost best CV ROC AUC:", catboost_search.best_score_)
print("CatBoost best params:", catboost_search.best_params_)


## Model Comparision

In [ ]:
from sklearn.metrics import accuracy_score, roc_auc_score, precision_score, recall_score, f1_score

candidates = {
    "logistic_regression": lr_search.best_estimator_,
    "random_forest": rf_search.best_estimator_,
    "gradient_boosting": gb_search.best_estimator_,
    "adaboost": adaboost_search.best_estimator_,
    "xgboost": xgb_search.best_estimator_,
    "catboost": catboost_search.best_estimator_,
}

comparison_rows = []
for name, estimator in candidates.items():
    y_pred = estimator.predict(X_test)
    y_proba = estimator.predict_proba(X_test)[:, 1]
    comparison_rows.append({
        "model": name,
        "accuracy": accuracy_score(y_test, y_pred),
        "roc_auc": roc_auc_score(y_test, y_proba),
        "precision": precision_score(y_test, y_pred),
        "recall": recall_score(y_test, y_pred),
        "f1_score": f1_score(y_test, y_pred),
    })

comparison_df = pd.DataFrame(comparison_rows).set_index("model")
comparison_df


# Save All Models Files to get Pickle file

In [ ]:
model_paths = {}

for name, estimator in candidates.items():
    path = MODEL_DIR / f"{name}.pkl"
    joblib.dump(estimator, path)
    model_paths[name] = path
    print(f"Saved {name} pipeline to: {path}")


## Sanity Check

In [ ]:
missing = [str(MODEL_DIR / f"{name}.pkl") for name in candidates if not (MODEL_DIR / f"{name}.pkl").exists()]
assert not missing, f"Missing pickle files: {missing}"
print(f"Verified all {len(candidates)} pickle files exist under {MODEL_DIR}")

loaded_pipelines = {name: joblib.load(MODEL_DIR / f"{name}.pkl") for name in candidates}
for name, pipeline in loaded_pipelines.items():
    preds = pipeline.predict(X_test.head(5))
    print(f"{name}: loaded OK, sample predictions = {preds}")


## Model Evaluation

In [ ]:
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, confusion_matrix, roc_curve, precision_recall_curve,
    average_precision_score, brier_score_loss,
)
from sklearn.model_selection import StratifiedKFold
from sklearn.utils.class_weight import compute_sample_weight
from sklearn.calibration import calibration_curve
from sklearn.base import clone
import matplotlib.pyplot as plt


eval_pipelines = {name: joblib.load(MODEL_DIR / f"{name}.pkl") for name in MODEL_NAMES}
print("loaded pipelines from disk:", list(eval_pipelines.keys()))


In [ ]:
eval_results = []
pred_test = {}
proba_test = {}
cm_test = {}

for name, pipe in eval_pipelines.items():
    y_pred = pipe.predict(X_test)
    y_proba = pipe.predict_proba(X_test)[:, 1]
    pred_test[name] = y_pred
    proba_test[name] = y_proba
    cm_test[name] = confusion_matrix(y_test, y_pred)

    eval_results.append({
        "model": name,
        "accuracy": accuracy_score(y_test, y_pred),
        "precision": precision_score(y_test, y_pred),
        "recall": recall_score(y_test, y_pred),
        "f1": f1_score(y_test, y_pred),
        "roc_auc": roc_auc_score(y_test, y_proba),
    })

eval_df = pd.DataFrame(eval_results).set_index("model")
eval_df


In [ ]:
# verify recomputed metrics against the evaluation set
verification = comparison_df[["accuracy", "precision", "recall", "f1_score"]].rename(
    columns={"accuracy": "dev_accuracy", "precision": "dev_precision",
             "recall": "dev_recall", "f1_score": "dev_f1"}
).join(eval_df[["accuracy", "precision", "recall", "f1"]])

verification["accuracy_diff"] = verification["accuracy"] - verification["dev_accuracy"]
verification["precision_diff"] = verification["precision"] - verification["dev_precision"]
verification["recall_diff"] = verification["recall"] - verification["dev_recall"]
verification["f1_diff"] = verification["f1"] - verification["dev_f1"]

max_abs_diff = verification[["accuracy_diff", "precision_diff", "recall_diff", "f1_diff"]].abs().max().max()
print("max absolute diff vs model-developer's reported numbers:", max_abs_diff)
verification[["accuracy_diff", "precision_diff", "recall_diff", "f1_diff"]]


In [ ]:


fig, axes = plt.subplots(2, 3, figsize=(15, 9))
for ax, name in zip(axes.flat, MODEL_NAMES):
    ConfusionMatrixDisplay(cm_test[name], display_labels=["Not readmitted", "Readmitted"]).plot(
        ax=ax, colorbar=False, values_format="d"
    )
    ax.set_title(name)
plt.tight_layout()
plt.show()

for name in MODEL_NAMES:
    tn, fp, fn, tp = cm_test[name].ravel()
    print(f"{name}: TN={tn} FP={fp} FN={fn} TP={tp}")


In [ ]:
plt.figure(figsize=(7, 6))
for name in MODEL_NAMES:
    fpr, tpr, _ = roc_curve(y_test, proba_test[name])
    plt.plot(fpr, tpr, label=f"{name} (AUC={eval_df.loc[name, 'roc_auc']:.3f})")
plt.plot([0, 1], [0, 1], "k--", label="chance (AUC=0.500)")
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("ROC curves — held-out test set")
plt.legend(loc="lower right")
plt.show()


In [ ]:
plt.figure(figsize=(7, 6))
ap_scores = {}
for name in MODEL_NAMES:
    precision_curve, recall_curve, _ = precision_recall_curve(y_test, proba_test[name])
    ap_scores[name] = average_precision_score(y_test, proba_test[name])
    plt.plot(recall_curve, precision_curve, label=f"{name} (AP={ap_scores[name]:.3f})")
positive_rate = y_test.mean()
plt.axhline(positive_rate, color="k", linestyle="--", label=f"no-skill baseline (AP={positive_rate:.3f})")
plt.xlabel("Recall")
plt.ylabel("Precision")
plt.title("Precision-Recall curves — held-out test set")
plt.legend(loc="lower left")
plt.show()


In [ ]:
# Evaluate the majority-class-only baseline for the test set
majority_class = int(y_test.value_counts().idxmax())
baseline_accuracy = (y_test == majority_class).mean()
baseline_f1 = f1_score(y_test, np.full(len(y_test), majority_class))
baseline_recall = recall_score(y_test, np.full(len(y_test), majority_class))

print("test class distribution:", y_test.value_counts(normalize=True).to_dict())
print(f"majority-class-only baseline: accuracy={baseline_accuracy:.4f}, f1={baseline_f1:.4f}, recall={baseline_recall:.4f}")
print(eval_df[["accuracy", "f1", "recall"]])


In [ ]:
top3_names = eval_df.sort_values("f1", ascending=False).head(3).index.tolist()
print("top 3 candidates by recomputed hold-out F1:", top3_names)
eval_df.loc[top3_names]


In [ ]:
# stratified 5-fold CV on the training pool for the top 3 
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
cv_records = []

for name in top3_names:
    base_classifier = candidates[name].named_steps["clf"]
    fold_scores = {"accuracy": [], "precision": [], "recall": [], "f1": [], "roc_auc": []}

    for train_idx, val_idx in skf.split(X_train, y_train):
        X_fold_train, X_fold_val = X_train.iloc[train_idx], X_train.iloc[val_idx]
        y_fold_train, y_fold_val = y_train.iloc[train_idx], y_train.iloc[val_idx]
        fold_weight = compute_sample_weight(class_weight="balanced", y=y_fold_train)

        fold_pipeline = Pipeline(steps=[
            ("preprocessor", clone(preprocessor)),
            ("classifier", clone(base_classifier)),
        ])
        fold_pipeline.fit(X_fold_train, y_fold_train, classifier__sample_weight=fold_weight)

        fold_pred = fold_pipeline.predict(X_fold_val)
        fold_proba = fold_pipeline.predict_proba(X_fold_val)[:, 1]

        fold_scores["accuracy"].append(accuracy_score(y_fold_val, fold_pred))
        fold_scores["precision"].append(precision_score(y_fold_val, fold_pred))
        fold_scores["recall"].append(recall_score(y_fold_val, fold_pred))
        fold_scores["f1"].append(f1_score(y_fold_val, fold_pred))
        fold_scores["roc_auc"].append(roc_auc_score(y_fold_val, fold_proba))

    record = {"model": name}
    for metric, values in fold_scores.items():
        record[f"cv_{metric}_mean"] = np.mean(values)
        record[f"cv_{metric}_std"] = np.std(values)
    cv_records.append(record)

cv_df = pd.DataFrame(cv_records).set_index("model")
cv_df


In [ ]:
# decision-threshold analysis on the test set probabilities for the top 3 candidates
thresholds = np.arange(0.10, 0.95, 0.05)
threshold_records = []
for name in top3_names:
    proba = proba_test[name]
    for t in thresholds:
        pred_t = (proba >= t).astype(int)
        threshold_records.append({
            "model": name,
            "threshold": round(t, 2),
            "precision": precision_score(y_test, pred_t, zero_division=0),
            "recall": recall_score(y_test, pred_t, zero_division=0),
            "f1": f1_score(y_test, pred_t, zero_division=0),
        })
threshold_df = pd.DataFrame(threshold_records)

fig, axes = plt.subplots(1, 3, figsize=(16, 4.5), sharey=True)
for ax, name in zip(axes, top3_names):
    sub = threshold_df[threshold_df["model"] == name]
    ax.plot(sub["threshold"], sub["precision"], label="precision")
    ax.plot(sub["threshold"], sub["recall"], label="recall")
    ax.plot(sub["threshold"], sub["f1"], label="f1")
    ax.axvline(0.5, color="grey", linestyle=":", label="default 0.5")
    ax.set_title(name)
    ax.set_xlabel("threshold")
axes[0].set_ylabel("score")
axes[0].legend()
plt.tight_layout()
plt.show()

best_f1_rows = threshold_df.loc[threshold_df.groupby("model")["f1"].idxmax()]
best_f1_rows


In [ ]:
# calibration check for the top 3 candidates on the test set
fig, axes = plt.subplots(1, 3, figsize=(16, 4.5), sharey=True, sharex=True)
brier_scores = {}
for ax, name in zip(axes, top3_names):
    proba = proba_test[name]
    frac_pos, mean_pred = calibration_curve(y_test, proba, n_bins=10)
    brier_scores[name] = brier_score_loss(y_test, proba)
    ax.plot(mean_pred, frac_pos, marker="o", label=name)
    ax.plot([0, 1], [0, 1], "k--", label="perfectly calibrated")
    ax.set_title(f"{name} (Brier={brier_scores[name]:.4f})")
    ax.set_xlabel("mean predicted probability")
axes[0].set_ylabel("observed fraction positive")
axes[0].legend()
plt.tight_layout()
plt.show()
print("Brier scores (lower is better):", brier_scores)


In [ ]:
# final consolidated comparison table: hold-out metrics (all 6) + CV / threshold / calibration (top 3 only)
final_summary_df = eval_df.copy()
final_summary_df["cv_f1_mean"] = cv_df["cv_f1_mean"]
final_summary_df["cv_f1_std"] = cv_df["cv_f1_std"]
final_summary_df["cv_roc_auc_mean"] = cv_df["cv_roc_auc_mean"]
final_summary_df["best_threshold_f1"] = best_f1_rows.set_index("model")["f1"]
final_summary_df["brier_score"] = pd.Series(brier_scores)

final_summary_df


## Inference

Model Selected: gradient_boosting

In [ ]:
gradient_boosting_pipeline = joblib.load(MODEL_DIR / "gradient_boosting.pkl")

readmission_risk_score = gradient_boosting_pipeline.predict_proba(X)[:, 1]
print("scored patients:", len(readmission_risk_score))
print(f"score range: min={readmission_risk_score.min():.4f}, max={readmission_risk_score.max():.4f}, mean={readmission_risk_score.mean():.4f}")

In [ ]:
risk_low_cut, risk_high_cut = np.quantile(readmission_risk_score, [1 / 3, 2 / 3])
print(f"tertile cut points on observed score distribution: low/medium={risk_low_cut:.4f}, medium/high={risk_high_cut:.4f}")

risk_segment = pd.cut(
    readmission_risk_score,
    bins=[-np.inf, risk_low_cut, risk_high_cut, np.inf],
    labels=["low", "medium", "high"],
)
print(pd.Series(risk_segment).value_counts())

In [ ]:
risk_table = pd.DataFrame({
    "patient_id": df["patient_id"].values,
    "readmission_risk_score": readmission_risk_score,
    "risk_segment": risk_segment,
    "readmission_flag": df[TARGET].values,
})
print(risk_table.shape)
risk_table.head()

In [ ]:
segment_summary = risk_table.groupby("risk_segment", observed=True).agg(
    n_patients=("patient_id", "count"),
    observed_readmission_rate=("readmission_flag", "mean"),
)
segment_summary["pct_of_patients"] = (segment_summary["n_patients"] / len(risk_table) * 100).round(2)
segment_summary["observed_readmission_rate"] = segment_summary["observed_readmission_rate"].round(4)
segment_summary = segment_summary.loc[["low", "medium", "high"]]
segment_summary

In [ ]:
# persist the risk-scored table in domain1/temp
risk_table.to_csv(RISK_TABLE_PATH, index=False)
print("saved risk-scored table to:", RISK_TABLE_PATH.resolve())

## Model Testing


In [ ]:
test_results = {}

# schema check 1: a dataframe missing an expected feature column must fail, and not predict
X_missing_col = X_test.drop(columns=[feature_cols[0]]).head(3)
try:
    gradient_boosting_pipeline.predict_proba(X_missing_col)
    test_results["schema_missing_column_raises"] = False
except Exception as e:
    test_results["schema_missing_column_raises"] = True
    print("missing-column input raised:", type(e).__name__, "-", str(e)[:150])

# schema check 2: Small-batch inputs must produce valid predictions
single_row = X_test.head(1)
small_batch = X_test.head(5)
single_proba = gradient_boosting_pipeline.predict_proba(single_row)[:, 1]
batch_proba = gradient_boosting_pipeline.predict_proba(small_batch)[:, 1]
test_results["schema_single_row_ok"] = bool(single_proba.shape == (1,) and np.isfinite(single_proba).all())
test_results["schema_small_batch_ok"] = bool(batch_proba.shape == (5,) and np.isfinite(batch_proba).all())

print("single-row proba:", single_proba)
print("small-batch proba:", batch_proba)
for k in ["schema_missing_column_raises", "schema_single_row_ok", "schema_small_batch_ok"]:
    print(f"{k}: {'PASS' if test_results[k] else 'FAIL'}")


In [ ]:
# missing-value: NaN in a real numeric feature, rows copied from the real feature space
numeric_test_col = "hemoglobin" if "hemoglobin" in numeric_cols else numeric_cols[0]
nan_batch = X_test.head(4).copy()
nan_batch.loc[nan_batch.index[0], numeric_test_col] = np.nan

try:
    nan_proba = gradient_boosting_pipeline.predict_proba(nan_batch)[:, 1]
    nan_no_error = True
except Exception as e:
    nan_no_error = False
    print("NaN input raised unexpectedly:", type(e).__name__, "-", str(e)[:150])

if nan_no_error:
    print(f"NaN injected in '{numeric_test_col}' -> proba:", nan_proba)
    nan_in_range = bool(np.all((nan_proba >= 0) & (nan_proba <= 1)) and np.isfinite(nan_proba).all())
else:
    nan_in_range = False

test_results["missing_value_imputer_handles_nan"] = nan_no_error and nan_in_range
print("missing_value_imputer_handles_nan:", "PASS" if test_results["missing_value_imputer_handles_nan"] else "FAIL")


In [ ]:
# unseen categorical-level edge case: a category value never seen in training
unseen_col = "gender"
unseen_row = X_test.head(1).copy()
unseen_row.loc[unseen_row.index[0], unseen_col] = "UNSEEN_TEST_CATEGORY"

try:
    unseen_proba = gradient_boosting_pipeline.predict_proba(unseen_row)[:, 1]
    unseen_no_error = True
except Exception as e:
    unseen_no_error = False
    print("unseen-category input raised unexpectedly:", type(e).__name__, "-", str(e)[:150])

if unseen_no_error:
    print(f"unseen '{unseen_col}' value -> proba:", unseen_proba)
    unseen_in_range = bool(np.all((unseen_proba >= 0) & (unseen_proba <= 1)))
else:
    unseen_in_range = False

test_results["unseen_category_handled"] = unseen_no_error and unseen_in_range
print("unseen_category_handled:", "PASS" if test_results["unseen_category_handled"] else "FAIL")


In [ ]:
# output sanity 1: probability is within [0,1]
test_results["all_probabilities_in_range"] = bool(
    (readmission_risk_score >= 0).all() and (readmission_risk_score <= 1).all()
)
print("probability range check: min=", readmission_risk_score.min(), "max=", readmission_risk_score.max())
print("all_probabilities_in_range:", "PASS" if test_results["all_probabilities_in_range"] else "FAIL")

# output sanity 2: risk segment counts are reproducible
recomputed_low_cut, recomputed_high_cut = np.quantile(readmission_risk_score, [1 / 3, 2 / 3])
recomputed_segment = pd.cut(
    readmission_risk_score,
    bins=[-np.inf, recomputed_low_cut, recomputed_high_cut, np.inf],
    labels=["low", "medium", "high"],
)
thresholds_match = bool(np.isclose(recomputed_low_cut, risk_low_cut) and np.isclose(recomputed_high_cut, risk_high_cut))
counts_match = pd.Series(recomputed_segment).value_counts().equals(pd.Series(risk_segment).value_counts())
test_results["risk_segments_reproducible"] = bool(thresholds_match and counts_match)
print("recomputed cuts:", recomputed_low_cut, recomputed_high_cut, "vs stored:", risk_low_cut, risk_high_cut)
print("recomputed counts:\n", pd.Series(recomputed_segment).value_counts())
print("risk_segments_reproducible:", "PASS" if test_results["risk_segments_reproducible"] else "FAIL")

# output sanity 3: no duplicate patient_id in the persisted deliverable CSV
persisted = pd.read_csv(RISK_TABLE_PATH)
n_dupes = int(persisted["patient_id"].duplicated().sum())
test_results["no_duplicate_patient_ids"] = bool(n_dupes == 0)
print("duplicate patient_id count in persisted CSV:", n_dupes)
print("no_duplicate_patient_ids:", "PASS" if test_results["no_duplicate_patient_ids"] else "FAIL")


In [ ]:
summary_df = pd.DataFrame({
    "check": list(test_results.keys()),
    "status": ["PASS" if v else "FAIL" for v in test_results.values()],
})
print(summary_df.to_string(index=False))

overall_pass = all(test_results.values())
print("\nOverall go/no-go:", "GO" if overall_pass else "NO-GO")
